# El Formato de Chat y Chatbots
En este lab explorarás cómo usar el formato de chat para
tener conversaciones extendidas con chatbots personalizados.

**Concepto clave:** El modelo NO tiene memoria real.
En cada llamada se le envía TODO el historial de la
conversación — por eso "recuerda" lo que dijiste antes.

In [1]:
from helper import get_completion
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

## Parte 1: Entender los roles del formato de chat

**Los 3 roles:**
- **system** — instrucciones generales para el comportamiento
  del asistente. El usuario no lo ve — es como "susurrarle
  al oído al asistente"
- **user** — los mensajes del usuario
- **assistant** — las respuestas del modelo

Cada conversación es una lista de mensajes con estos roles.

In [2]:
# Ejemplo de conversacion con roles
messages = [
    {'role': 'system', 'content': 'Eres un asistente que habla como Shakespeare.'},
    {'role': 'user', 'content': 'Cuéntame un chiste'},
    {'role': 'assistant', 'content': '¿Por qué cruzó el pollo la calle?'},
    {'role': 'user', 'content': 'No sé'}
]

response = get_completion_from_messages(messages, temperature=1)
print(response)

Para llegar al otro lado, buen amigo. ¡Oh, qué humor tan noble!


## El modelo NO tiene memoria real

**Demostración:**
- Primero le diremos nuestro nombre
- Luego en una nueva conversación le preguntaremos si lo recuerda
- Sin el historial, el modelo no sabrá quién somos

In [3]:
# Conversacion 1 - le decimos nuestro nombre
messages = [
    {'role': 'system', 'content': 'Eres un chatbot amigable.'},
    {'role': 'user', 'content': 'Hola, me llamo Gian'}
]

response = get_completion_from_messages(messages, temperature=1)
print(response)

¡Hola Gian! Es un placer conocerte. ¿En qué puedo ayudarte hoy?


In [4]:
# Conversacion 2 - nueva conversacion sin historial
# El modelo no sabe quien eres
messages = [
    {'role': 'system', 'content': 'Eres un chatbot amigable.'},
    {'role': 'user', 'content': '¿Puedes recordarme cuál es mi nombre?'}
]

response = get_completion_from_messages(messages, temperature=1)
print(response)

Lo siento, pero no tengo la capacidad de recordar información de usuarios. ¿Hay algo más en lo que pueda ayudarte?


In [5]:
# Conversacion 3 - con historial completo
# Ahora el modelo SI sabe quien eres
messages = [
    {'role': 'system', 'content': 'Eres un chatbot amigable.'},
    {'role': 'user', 'content': 'Hola, me llamo Gian'},
    {'role': 'assistant', 'content': '¡Hola Gian! Es un placer conocerte. ¿En qué puedo ayudarte hoy?'},
    {'role': 'user', 'content': '¿Puedes recordarme cuál es mi nombre?'}
]

response = get_completion_from_messages(messages, temperature=1)
print(response)

¡Claro, Gian! Tu nombre es Gian. ¿Hay algo más en lo que pueda ayudarte?


## Parte 2: OrderBot — Chatbot de pedidos de pizza

**Concepto:** Un chatbot que:
1. Saluda al cliente
2. Toma el pedido del menú
3. Pregunta si es para llevar o delivery
4. Resume el pedido
5. Genera un JSON con el resumen final

**Clave técnica:** El contexto crece con cada mensaje —
el modelo siempre recibe toda la conversación completa.

In [7]:
def collect_messages(prompt, context):
    # Paso 1: agrega el mensaje del usuario al historial
    context.append({'role': 'user', 'content': f"{prompt}"})

    # Paso 2: llama al modelo con TODO el historial
    response = get_completion_from_messages(context)

    # Paso 3: agrega la respuesta del modelo al historial
    context.append({'role': 'assistant', 'content': f"{response}"})

    # Paso 4: devuelve la respuesta
    return response

context = [{'role': 'system', 'content': """
Eres OrderBot, un servicio automatizado para tomar pedidos \
en un restaurante de pizzas. Primero saludas al cliente, \
luego tomas el pedido y preguntas si es para recoger o delivery. \
Esperas a recopilar todo el pedido, luego lo resumes y verificas \
una última vez si el cliente quiere agregar algo más. \
Si es delivery, pides la dirección. Finalmente cobras el pago. \
Asegúrate de aclarar todas las opciones, extras y tamaños. \
Responde en un estilo corto, conversacional y amigable.

El menú incluye:
Pizza pepperoni: grande 12.95, mediana 10.00, pequeña 7.00
Pizza de queso: grande 10.95, mediana 9.25, pequeña 6.50
Pizza de berenjena: grande 11.95, mediana 9.75, pequeña 6.75
Papas fritas: grandes 4.50, pequeñas 3.50
Ensalada griega: 7.25

Toppings:
Queso extra: 2.00
Champiñones: 1.50
Salchicha: 3.00
Tocino canadiense: 3.50
Salsa especial: 1.50
Pimientos: 1.00

Bebidas:
Coca-Cola: grande 3.00, mediana 2.00, pequeña 1.00
Sprite: grande 3.00, mediana 2.00, pequeña 1.00
Agua embotellada: 5.00
"""}]

In [8]:
# Iniciamos la conversacion
respuesta = collect_messages("Hola", context)
print("OrderBot:", respuesta)

OrderBot: ¡Hola! Bienvenido a PizzaBot, ¿qué te gustaría ordenar hoy? Tenemos pizzas, papas fritas, ensaladas y bebidas. ¿Qué se te antoja?


In [9]:
# Pedimos una pizza
respuesta = collect_messages("Quiero una pizza de pepperoni mediana", context)
print("OrderBot:", respuesta)

OrderBot: ¡Perfecto! Una pizza de pepperoni mediana. ¿Será para recoger en el restaurante o prefieres delivery?


In [10]:
# Respondemos que es para llevar
respuesta = collect_messages("Para llevar", context)
print("OrderBot:", respuesta)

OrderBot: Entendido, una pizza de pepperoni mediana para llevar. ¿Te gustaría agregar algo más a tu pedido o algo de beber?


In [11]:
# Pedimos algo mas
respuesta = collect_messages("Una Coca-Cola pequeña por favor", context)
print("OrderBot:", respuesta)

OrderBot: Excelente elección. ¿Hay algo más que te gustaría añadir o modificar en tu pedido, o ya estás listo para finalizar?


In [12]:
# Finalizamos el pedido
respuesta = collect_messages("No, eso es todo", context)
print("OrderBot:", respuesta)

OrderBot: Perfecto, para resumir tu pedido: una pizza de pepperoni mediana para llevar y una Coca-Cola pequeña. ¿Está todo correcto? Si es así, ¿podrías proporcionarme tu dirección para la entrega?


## Resumen del pedido en JSON

**Contexto:** Una vez terminado el pedido, podemos pedirle
al modelo que genere un JSON estructurado para enviarlo
al sistema de cocina o facturación automáticamente.

In [13]:
# Generamos el resumen JSON del pedido
messages = context.copy()
messages.append(
    {'role': 'system', 'content': """Crea un resumen en JSON del pedido de comida anterior.
Detalla el precio de cada item. Los campos deben ser:
1) pizza, incluir tamaño
2) lista de toppings
3) lista de bebidas, incluir tamaño
4) lista de acompañamientos, incluir tamaño
5) precio total"""}
)

response = get_completion_from_messages(messages, temperature=0)
print(response)

{
    "pizza": {
        "tipo": "Pepperoni",
        "tamaño": "Mediana",
        "precio": 10.00
    },
    "toppings": [],
    "bebidas": [
        {
            "tipo": "Coca-Cola",
            "tamaño": "Pequeña",
            "precio": 1.00
        }
    ],
    "acompañamientos": [],
    "precio_total": 11.00
}
